# <font color="#418FDE" size="6.5" uppercase>**Naive Bayes Models**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Match Naive Bayes variants to continuous, count, binary, and categorical feature representations. 
- Configure smoothing, priors, and sample weights to improve probabilistic classification behavior. 
- Use partial_fit for incremental Naive Bayes training on batches. 


## **1. Naive Bayes Assumptions**

### **1.1. Continuous Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_01.jpg?v=1783847484" width="250">



>* Continuous features are real-valued measurements.
>* Classes model values by center and spread.

>* Naive Bayes assumes features act independently.
>* Continuous values should follow smooth class distributions.

>* Scores typical continuous values higher by class.
>* Use for numeric features, not counts/categories.



In [ ]:
#@title Python Code - Continuous Features

# Continuous values need distribution based assumptions.
# Gaussian Naive Bayes fits numeric measurements.
# This example compares class specific likelihoods.

import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two continuous feature groups.
normal = rng.normal(loc=[5.0, 2.0], scale=[0.6, 0.3], size=(12, 2))
faulty = rng.normal(loc=[7.0, 3.5], scale=[0.5, 0.4], size=(12, 2))

# Stack data and class labels.
X = np.vstack([normal, faulty])
y = np.array([0] * 12 + [1] * 12)

# Check the tiny dataset shape.
assert X.shape == (24, 2), "Unexpected feature matrix shape."

# Compute class means for features.
mean_0 = X[y == 0].mean(axis=0)
mean_1 = X[y == 1].mean(axis=0)

# Compute class variances safely.
var_0 = X[y == 0].var(axis=0) + 1e-9
var_1 = X[y == 1].var(axis=0) + 1e-9

# Define a Gaussian log density.
def gaussian_logpdf(x, mean, var):

    # Return summed feature log likelihood.
    return (-0.5 * np.log(2 * np.pi * var)
            -0.5 * ((x - mean) ** 2) / var).sum()

# Choose one new continuous measurement.
new_point = np.array([6.8, 3.2])

# Score the point under each class.
score_0 = gaussian_logpdf(new_point, mean_0, var_0)
score_1 = gaussian_logpdf(new_point, mean_1, var_1)

# Pick the more likely class.
predicted = int(score_1 > score_0)
class_name = ["normal", "faulty"][predicted]

# Print a short teaching summary.
print("Feature type: continuous measurements")
print("Model match: Gaussian Naive Bayes idea")
print("Normal mean:", np.round(mean_0, 2))
print("Faulty mean:", np.round(mean_1, 2))
print("New point:", new_point)
print("Log score, normal:", round(score_0, 2))
print("Log score, faulty:", round(score_1, 2))
print("Predicted class:", class_name)



### **1.2. Continuous Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_02.jpg?v=1783847488" width="250">



>* Continuous features use class-specific probability distributions.
>* Gaussian Naive Bayes models separate feature likelihoods.

>* Gaussian Naive Bayes suits smooth measurements.
>* Skewed or multimodal features may need alternatives.

>* Continuous Naive Bayes simplifies numeric measurements.
>* Best when features truly behave continuously.



In [ ]:
#@title Python Code - Continuous Features

# Continuous values need smooth probability assumptions.
# Gaussian style summaries fit many measurements.
# This example compares classwise numeric patterns.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two classes of heights.
class_a = rng.normal(loc=165, scale=6, size=80)
class_b = rng.normal(loc=180, scale=6, size=80)

# Check the sample sizes.
assert class_a.size == 80 and class_b.size == 80

# Summarize each class separately.
mean_a, std_a = class_a.mean(), class_a.std(ddof=1)
mean_b, std_b = class_b.mean(), class_b.std(ddof=1)

# Print short teaching takeaways.
print(f"Class A mean height: {mean_a:.1f} cm")
print(f"Class A spread: {std_a:.1f} cm")
print(f"Class B mean height: {mean_b:.1f} cm")
print(f"Class B spread: {std_b:.1f} cm")

# Build smooth x values.
x = np.linspace(145, 200, 300)

# Define a Gaussian density helper.
def gaussian_pdf(x_values, mean_value, std_value):
    scale = std_value * np.sqrt(2 * np.pi)
    exponent = -0.5 * ((x_values - mean_value) / std_value) ** 2
    return np.exp(exponent) / scale

# Compute classwise smooth densities.
pdf_a = gaussian_pdf(x, mean_a, std_a)
pdf_b = gaussian_pdf(x, mean_b, std_b)

# Plot continuous feature distributions.
plt.figure(figsize=(7, 4))
plt.hist(class_a, bins=12, density=True, alpha=0.45, label="Class A")
plt.hist(class_b, bins=12, density=True, alpha=0.45, label="Class B")

# Overlay smooth Gaussian curves.
plt.plot(x, pdf_a, linewidth=2, label="A Gaussian fit")
plt.plot(x, pdf_b, linewidth=2, label="B Gaussian fit")

# Add labels and explanation.
plt.title("Continuous feature: height by class")
plt.xlabel("Height in centimeters")
plt.ylabel("Estimated probability density")

# Finish the beginner friendly plot.
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Feature Type Matching**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_03.jpg?v=1783847489" width="250">



>* Choose variants by feature representation.
>* Match continuous, count, binary, categorical features.

>* Match variants to actual feature meaning.
>* Mixed feature types need careful modeling.

>* Encoding changes how features should be modeled.
>* Match Naive Bayes to feature representation.



In [ ]:
#@title Python Code - Feature Type Matching

# Match Naive Bayes variants to feature types.
# This example uses tiny hand made data.
# We compare assumptions, not model training.

# Import only small numerical tools.
import numpy as np
import pandas as pd

# Build tiny examples for each feature style.
continuous = np.array([5.1, 5.9, 6.3])
counts = np.array([0, 3, 7])
binary = np.array([0, 1, 1])
categories = pd.Series(["Chrome", "Safari", "Chrome"])

# Summarize each representation clearly.
examples = {
    "GaussianNB": "continuous measurements",
    "MultinomialNB": "nonnegative counts",
    "BernoulliNB": "binary indicators",
    "CategoricalNB": "category codes",
}

# Show why the same topic can differ.
text_count = np.array([0, 2, 5])
text_presence = (text_count > 0).astype(int)

# Print a compact teaching summary.
print("Feature matching for Naive Bayes:")
print(f"GaussianNB -> {examples['GaussianNB']}")
print(f"MultinomialNB -> {examples['MultinomialNB']}")
print(f"BernoulliNB -> {examples['BernoulliNB']}")
print(f"CategoricalNB -> {examples['CategoricalNB']}")

# Print tiny data examples.
print(f"Continuous example: {continuous.tolist()}")
print(f"Count example: {counts.tolist()}")
print(f"Binary example: {binary.tolist()}")
print(f"Categorical example: {categories.tolist()}")

# Show one raw source, two encodings.
print(f"Text as counts: {text_count.tolist()}")
print(f"Text as presence: {text_presence.tolist()}")



## **2. Naive Bayes Tuning**

### **2.1. Smoothing and Priors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_01.jpg?v=1783847478" width="250">



>* Smoothing prevents zero probabilities for unseen features.
>* It supports cautious predictions with limited data.

>* Priors set baseline class expectations.
>* Adjust priors when training data misleads.

>* Smoothing and priors jointly control confidence.
>* Balanced settings improve reliable probability estimates.



### **2.2. Priors and Smoothing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_02.jpg?v=1783847480" width="250">



>* Priors set baseline class expectations.
>* Smoothing prevents extreme unseen-feature probabilities.

>* Priors correct unrealistic class prevalence estimates.
>* Smoothing stabilizes predictions from sparse features.

>* Priors set baseline; smoothing controls evidence sensitivity.
>* Align both for credible, useful probabilities.



In [ ]:
#@title Python Code - Priors and Smoothing

# Priors shape starting class beliefs.
# Smoothing avoids impossible zero probabilities.
# Sample weights can rebalance evidence.

import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import log_loss

# Fix randomness for repeatable results.
rng = np.random.default_rng(7)
classes = np.array([0, 1])

# Build tiny categorical training data.
X_train = np.array([[0], [0], [0], [1], [1], [1]])
y_train = np.array([0, 0, 0, 1, 1, 1])

# Build test cases with unseen category.
X_test = np.array([[0], [1], [2]])
y_test = np.array([0, 1, 1])

# Fit without smoothing for comparison.
nb_zero = CategoricalNB(alpha=0.0, force_alpha=True)
nb_zero.fit(X_train, y_train)

# Fit with smoothing for safer probabilities.
nb_smooth = CategoricalNB(alpha=1.0)
nb_smooth.fit(X_train, y_train)

# Fit with custom priors favoring class one.
nb_prior = CategoricalNB(alpha=1.0, class_prior=[0.2, 0.8])
nb_prior.fit(X_train, y_train)

# Reweight one class zero example heavily.
weights = np.array([3.0, 1.0, 1.0, 1.0, 1.0, 1.0])
nb_weighted = CategoricalNB(alpha=1.0)
nb_weighted.fit(X_train, y_train, sample_weight=weights)

# Collect probabilities for unseen category.
p_zero = nb_zero.predict_proba(X_test)[2]
p_smooth = nb_smooth.predict_proba(X_test)[2]
p_prior = nb_prior.predict_proba(X_test)[2]
p_weighted = nb_weighted.predict_proba(X_test)[0]

# Measure calibration style with log loss.
loss_smooth = log_loss(y_test, nb_smooth.predict_proba(X_test), labels=classes)
loss_prior = log_loss(y_test, nb_prior.predict_proba(X_test), labels=classes)

# Print short teaching summary.
print("Unseen category, alpha=0.0:", np.round(p_zero, 3))
print("Unseen category, alpha=1.0:", np.round(p_smooth, 3))
print("With custom priors:", np.round(p_prior, 3))
print("Weighted known category 0:", np.round(p_weighted, 3))
print("Log loss, smoothed model:", round(loss_smooth, 3))
print("Log loss, prior model:", round(loss_prior, 3))



### **2.3. Sample Weighting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_03.jpg?v=1783847482" width="250">



>* Weights make some examples count more.
>* Emphasize recent or higher-quality data.

>* Weighting helps correct class imbalance effects.
>* It improves rare-event probability estimates.

>* Use weights carefully to avoid unstable probabilities.
>* Weight trusted data to reflect real priorities.



## **3. Incremental Training Controls**

### **3.1. Batchwise Model Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_01.jpg?v=1783847471" width="250">



>* Naive Bayes learns gradually from data batches.
>* Updates counts without full retraining each time.

>* Batches build on earlier learning cumulatively.
>* Use representative batches early for balance.

>* Batch size balances responsiveness, cost, and stability.
>* Regular updates keep Naive Bayes current.



### **3.2. Streaming Batches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_02.jpg?v=1783847475" width="250">



>* Streaming batches update models as data arrives.
>* Useful for memory limits and fast adaptation.

>* Keep features and labels consistent across batches.
>* Moderate batch sizes balance noise and responsiveness.

>* Batch order affects adaptation to changing patterns.
>* Monitor representativeness to keep probabilities stable.



### **3.3. Partial Fit Batches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_03.jpg?v=1783847477" width="250">



>* Small batches update Naive Bayes incrementally.
>* Useful for streaming data and limited memory.

>* Choose balanced batch sizes for stable updates.
>* Early batches should include all classes.

>* Keep preprocessing consistent and monitor distribution shifts.
>* Incremental updates keep models current over time.



# <font color="#418FDE" size="6.5" uppercase>**Naive Bayes Models**</font>


In this lecture, you learned to:
- Match Naive Bayes variants to continuous, count, binary, and categorical feature representations. 
- Configure smoothing, priors, and sample weights to improve probabilistic classification behavior. 
- Use partial_fit for incremental Naive Bayes training on batches. 

In the next Lecture (Lecture B), we will go over 'Probability Calibration'